In [25]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt \
    python-dotenv email-validator pyngrok \
    fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib \
    transformers accelerate torch stopwordsiso
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 30.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_sent_ud_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [26]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [27]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

Overwriting db.py


In [28]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash) VALUES (%s,%s,%s)",
                    (username, email, hash_pw(pw)))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Overwriting auth.py


In [29]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Overwriting email_utils.py


In [37]:
%%writefile app.py
import os, re, requests, streamlit as st
from db import init_db
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

st.set_page_config(page_title="Employee Wellness Analytics", page_icon="🔐")

st.markdown("""
<style>

/* Main background */
.stApp {
    background-color: #FFFFFF;
}

/* Main Heading Title */
.main-title {
    font-size: 2.2rem !important;
    font-weight: 800 !important;
    color: #166534 !important;
    margin-bottom: 0.5rem;
}

/* Dashboard Title (Analysis Page) */
.dashboard-title {
    font-size: 2.2rem !important;
    font-weight: 800 !important;
    color: #166534 !important;
    margin-top: 10px !important;
    margin-bottom: 15px !important;
}

/* User Welcome Header Box */
.welcome-card {
    background-color: #F8FAFC;
    border: 1px solid #E2E8F0;
    border-radius: 12px;
    padding: 12px 20px;
    margin-bottom: 20px;
}

/* Unified Card Container (Matching image_feeabc.png) */
.custom-card {
    border: 1px solid #3B4252;
    border-radius: 12px;
    overflow: hidden;
    margin-top: 20px;
    margin-bottom: 25px;
    background-color: #FFFFFF;
}

.custom-card-header {
    background-color: #3B4252; /* Dark slate header */
    color: #22C55E; /* Bright green text */
    font-size: 1.3rem;
    font-weight: 700;
    padding: 14px 20px;
}

.custom-card-body {
    padding: 20px;
    background-color: #F8FAFC;
}

/* Custom Grid inside Cards */
.metrics-grid-4 {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 15px;
}

.metrics-grid-5 {
    display: grid;
    grid-template-columns: repeat(5, 1fr);
    gap: 15px;
}

.metric-item-label {
    font-size: 0.9rem;
    color: #475569;
    margin-bottom: 6px;
    font-weight: 500;
}

.metric-item-value {
    font-size: 1.8rem;
    font-weight: 700;
    color: #0F172A;
}

/* Light Green Buttons */
.stButton>button, div[data-testid="stFormSubmitButton"]>button {
    border-radius: 8px;
    height: 2.5rem;
    font-weight: 600;
    font-size: 0.95rem;
    color: #166534 !important;
    background-color: #86EFAC !important;
    border: 1px solid #4ADE80;
    transition: 0.2s ease-in-out;
}

.stButton>button:hover, div[data-testid="stFormSubmitButton"]>button:hover {
    background-color: #4ADE80 !important;
    border-color: #22C55E;
    color: #14532D !important;
}

/* File uploader styling */
[data-testid="stFileUploader"] {
    border: 1px dashed #CBD5E1;
    border-radius: 10px;
    padding: 10px;
    background: #F8FAFC;
}

/* Text inputs */
.stTextInput input {
    border-radius: 8px !important;
    background-color: #FFFFFF !important;
    color: #0F172A !important;
    border: 1px solid #CBD5E1 !important;
}

</style>
""", unsafe_allow_html=True)

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "login"
if "token" not in st.session_state: st.session_state.token = None
if "email" not in st.session_state: st.session_state.email = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "view" not in st.session_state: st.session_state.view = "upload"
if "analysis" not in st.session_state: st.session_state.analysis = None

def goto(p): st.session_state.page = p; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

def display_analysis(r):

  st.markdown('<div class="dashboard-title">🧠 NLP ANALYSIS DASHBOARD</div>', unsafe_allow_html=True)
  st.caption("Multilingual Sentiment & Emotion Analysis Report")
  st.divider()

  # 1. Overview & Classification Container Box
  st.markdown(f"""
  <div class="custom-card">
      <div class="custom-card-header">📌 Overview & Classification</div>
      <div class="custom-card-body">
          <div class="metrics-grid-4">
              <div>
                  <div class="metric-item-label">😊 Sentiment</div>
                  <div class="metric-item-value">{r["final_sentiment"]}</div>
              </div>
              <div>
                  <div class="metric-item-label">🎭 Emotion</div>
                  <div class="metric-item-value">{r["final_emotion"]}</div>
              </div>
              <div>
                  <div class="metric-item-label">🌍 Language</div>
                  <div class="metric-item-value">{r["detected_language"]}</div>
              </div>
              <div>
                  <div class="metric-item-label">📄 Type</div>
                  <div class="metric-item-value">{r["file_type"].upper()}</div>
              </div>
          </div>
      </div>
  </div>
  """, unsafe_allow_html=True)

  st.markdown("---")

  with st.expander("🧹 Cleaned Text"):
      st.write(r["cleaned_text"])
  with st.expander("📝 Sentences"):
      for i, s in enumerate(r["sentences"], 1):
          st.write(f"{i}. {s}")
  with st.expander("🔤 Tokens (Original vs Filtered)"):
      st.write("Original:", r["original_tokens"])
      st.write("Filtered:", r["filtered_tokens"])
  with st.expander("📚 Translation & Lemmatization"):
      st.write("**English translation:**", r["translated_text"])
      st.write("**Lemmatized:**", r["lemmatized_text"])

  emoji_text = ", ".join(r["emoji_list"]) if r["emoji_list"] else "No emojis detected"
  st.info(f"😀 **Emoji Extraction:** {emoji_text}")

  scores = r["sentiment_scores"]

  st.divider()

  st.subheader("📈 Sentiment Analysis")
  st.caption(f"Prediction: **{r['final_sentiment']}**")

  st.bar_chart({
      "Positive": scores["pos"],
      "Neutral": scores["neu"],
      "Negative": scores["neg"],
  })

  st.metric("Compound Score", f"{scores['compound']:.3f}")

  st.divider()

  st.subheader("😊 Emotion Analysis")
  st.caption(f"Dominant Emotion: **{r['final_emotion']}**")

  st.bar_chart(r["emotion_scores"])

  # 2. Processing Statistics Container Box
  st.markdown(f"""
  <div class="custom-card">
      <div class="custom-card-header">📊 Processing Statistics</div>
      <div class="custom-card-body">
          <div class="metrics-grid-5">
              <div>
                  <div class="metric-item-label">Characters</div>
                  <div class="metric-item-value">{r["original_char_count"]}</div>
              </div>
              <div>
                  <div class="metric-item-label">Sentences</div>
                  <div class="metric-item-value">{len(r["sentences"])}</div>
              </div>
              <div>
                  <div class="metric-item-label">Tokens</div>
                  <div class="metric-item-value">{len(r["original_tokens"])}</div>
              </div>
              <div>
                  <div class="metric-item-label">Filtered</div>
                  <div class="metric-item-value">{len(r["filtered_tokens"])}</div>
              </div>
              <div>
                  <div class="metric-item-label">Emojis</div>
                  <div class="metric-item-value">{len(r["emoji_list"])}</div>
              </div>
          </div>
      </div>
  </div>
  """, unsafe_allow_html=True)


# ---- Logged-in view ----
if st.session_state.token:
    user = read_token(st.session_state.token)

    if user:

        # Header Box
        welcome_col, logout_col = st.columns([5, 1])
        with welcome_col:
            st.markdown(
                f'<div class="welcome-card"><b>WELCOME, {user["username"].upper()}!</b> ({user["email"]})</div>',
                unsafe_allow_html=True
            )
        with logout_col:
            if st.button("Log out"):
                st.session_state.token = None
                goto("login")

        # ================= ANALYSIS PAGE =================
        if st.session_state.view == "analysis":

            if st.button("⬅ Back to Upload"):
                st.session_state.view = "upload"
                st.rerun()

            display_analysis(st.session_state.analysis)
            st.stop()

        # ================= MAIN UPLOAD & CHAT VIEW =================


        st.markdown('<div class="main-title">🧠 Employee Wellness Analytics</div>', unsafe_allow_html=True)
        st.divider()

        # ---------------- 1. FILE UPLOADER  ----------------
        st.subheader("📁 Upload Employee Feedback")

        uploaded = st.file_uploader(
            "Choose a CSV or TXT file",
            type=["csv", "txt"],
        )

        headers = {
            "Authorization": f"Bearer {st.session_state.token}"
        }

        if uploaded is not None:

            is_csv = uploaded.name.lower().endswith(".csv")
            column_name = None

            if is_csv:
                column_name = (
                    st.text_input(
                        "Feedback column name (leave blank to use last column)"
                    ).strip()
                    or None
                )

            c1, c2 = st.columns(2)

            # ---------- Preview ----------
            if c1.button("Upload & Preview"):

                files = {
                    "file": (
                        uploaded.name,
                        uploaded.getvalue(),
                    )
                }

                try:
                    resp = requests.post(
                        f"{BACKEND_URL}/upload",
                        files=files,
                        headers=headers,
                        timeout=15,
                    )

                    if resp.status_code == 200:
                        data = resp.json()
                        st.success(
                            f"Uploaded '{data['filename']}' — {data['row_count']} rows."
                        )

                        if data["type"] == "csv":
                            st.write(
                                "**Columns:**",
                                ", ".join(data["columns"]),
                            )

                            if data["preview_rows"]:
                                st.dataframe(
                                    [
                                        dict(zip(data["columns"], row))
                                        for row in data["preview_rows"]
                                    ]
                                )

                        else:
                            st.text("\n".join(data["preview_lines"]))

                    else:
                        st.error("Upload failed.")

                except requests.exceptions.RequestException as e:
                    st.error(e)

            # ---------- Analysis ----------
            if c2.button("Run NLP Analysis"):

                files = {
                    "file": (
                        uploaded.name,
                        uploaded.getvalue(),
                    )
                }

                form = (
                    {"column": column_name}
                    if column_name
                    else {}
                )

                with st.spinner("🧠 Running multilingual NLP pipeline..."):

                    try:
                        resp = requests.post(
                            f"{BACKEND_URL}/analyze",
                            files=files,
                            data=form,
                            headers=headers,
                            timeout=120,
                        )

                        if resp.status_code == 200:
                            st.session_state.analysis = resp.json()
                            st.session_state.view = "analysis"
                            st.rerun()
                        else:
                            st.error("Analysis failed.")

                    except requests.exceptions.RequestException as e:
                        st.error(e)

        st.divider()

        # ---------------- 2. CHAT  ----------------
        st.subheader("💬 Wellness Assistant")
        st.caption(
            "A supportive space to talk about how you're feeling. "
            "Not a substitute for professional care."
        )

        chat_box = st.container(height=300)

        with chat_box:
            for turn in st.session_state.chat_history:
                with st.chat_message(turn["role"]):
                    st.write(turn["content"])

        user_msg = st.chat_input("How are you feeling today?")

        if user_msg:

            st.session_state.chat_history.append(
                {"role": "user", "content": user_msg}
            )

            recent_history = st.session_state.chat_history[-10:-1]

            try:
                resp = requests.post(
                    f"{BACKEND_URL}/chat",
                    json={
                        "message": user_msg,
                        "history": recent_history,
                    },
                    headers={
                        "Authorization": f"Bearer {st.session_state.token}"
                    },
                    timeout=60,
                )

                if resp.status_code == 200:
                    reply = resp.json()["reply"]
                else:
                    reply = "Sorry, I couldn't reach the wellness assistant."

            except requests.exceptions.RequestException:
                reply = "Sorry, I couldn't reach the wellness assistant."

            st.session_state.chat_history.append(
                {"role": "assistant", "content": reply}
            )

            st.rerun()

        if (
            st.session_state.chat_history
            and st.button("Clear chat")
        ):
            st.session_state.chat_history = []
            st.rerun()

        st.stop()

    st.session_state.token = None

# ---- Login ----
if st.session_state.page == "login":
    with st.form("login"):
        email = st.text_input("Email")
        pw = st.text_input("Password", type="password")
        go = st.form_submit_button("Log in")
    if go:
        user = get_user(email.strip().lower())
        if not user or not check_pw(pw, user["password_hash"]):
            st.error("Invalid email or password.")
        elif not user["is_verified"]:
            st.warning("Verify your email first.")
            st.session_state.email = user["email"]; goto("verify")
        else:
            st.session_state.token = make_token(user); st.rerun()

    c1, c2 = st.columns(2)
    if c1.button("Sign up"): goto("signup")
    if c2.button("Forgot password?"): goto("forgot")

# ---- Signup ----
elif st.session_state.page == "signup":
    with st.form("signup"):
        username = st.text_input("Username")
        email = st.text_input("Email")
        pw = st.text_input("Password", type="password")
        go = st.form_submit_button("Create account")
    if go:
        email = email.strip().lower()
        if len(username) < 3:
            st.error("Username too short.")
        elif not valid_pw(pw):
            st.error("Password needs 8+ chars, letters and numbers.")
        elif username_taken(username) or get_user(email):
            st.error("Username or email already in use.")
        else:
            create_user(username, email, pw)
            code = new_otp(); save_otp(email, code, "signup")
            ok, msg = send_otp(email, code, "signup")
            if ok:
                st.session_state.email = email
                st.success("Check your email for the code.")
                goto("verify")
            else:
                st.error(f"Email failed: {msg}")
    if st.button("← Back"): goto("login")

# ---- Verify OTP (signup) ----
elif st.session_state.page == "verify":
    email = st.session_state.email
    st.write(f"Enter the code sent to **{email}**")
    with st.form("verify"):
        code = st.text_input("Code", max_chars=6)
        go = st.form_submit_button("Verify")
    if go:
        if check_otp(email, code.strip(), "signup"):
            verify_user(email)
            st.success("Verified! Please log in.")
            goto("login")
        else:
            st.error("Invalid or expired code.")
    if st.button("← Back"): goto("login")

# ---- Forgot password ----
elif st.session_state.page == "forgot":
    with st.form("forgot"):
        email = st.text_input("Your account email")
        go = st.form_submit_button("Send reset code")
    if go:
        email = email.strip().lower()
        if get_user(email):
            code = new_otp(); save_otp(email, code, "password_reset")
            send_otp(email, code, "password_reset")
        st.session_state.email = email
        st.info("If that email exists, a code was sent.")
        goto("reset")
    if st.button("← Back"): goto("login")

# ---- Reset password ----
elif st.session_state.page == "reset":
    email = st.session_state.email
    with st.form("reset"):
        code = st.text_input("Reset code", max_chars=6)
        pw = st.text_input("New password", type="password")
        go = st.form_submit_button("Reset")
    if go:
        if not valid_pw(pw):
            st.error("Password needs 8+ chars, letters and numbers.")
        elif not check_otp(email, code.strip(), "password_reset"):
            st.error("Invalid or expired code.")
        else:
            set_password(email, pw)
            st.success("Password reset. Please log in.")
            goto("login")
    if st.button("← Back"): goto("login")

Overwriting app.py


In [31]:
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion (BERT).

Stopword filtering uses the `stopwordsiso` package, which ships stopword
sets for 50+ languages keyed by ISO 639-1 code (the same codes langdetect
returns), so any supported language is handled automatically instead of
needing a hardcoded list per language. If the detected language isn't in
stopwordsiso's coverage, filtering is simply skipped for that text.

Heavy libs (spacy model, translator, vader, BERT emotion model, Qwen chat
model) load once at import time via lazy module-level globals, so repeated
/analyze calls reuse them.
"""

import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
)
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Fine-tuned BERT model for emotion classification, trained on the
# GoEmotions dataset (28 fine-grained emotion labels). This directly
# matches the project brief's requirement for a "fine-tuned BERT model"
# for multi-label emotion classification.
BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

# Human-readable names for common languages this pipeline is likely to see.
# Purely cosmetic (shown in the UI) -- falls back to the raw ISO code if a
# language isn't listed here, so it never blocks processing of any language.
LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}


def _get_stopwords(language_code: str) -> set:
    """
    Returns the stopword set for `language_code` using stopwordsiso, which
    covers 50+ languages by ISO 639-1 code. Returns an empty set for any
    language it doesn't cover -- filtering is skipped rather than failing,
    so unsupported languages still flow through the rest of the pipeline.
    """
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

# Fixed label set the app's UI (charts, backend.py) is built around. Keeping
# this list stable means downstream code never has to change, no matter
# which underlying model produces the emotion.
EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]

EMOTION_EMOJI = {
    "Happy": "\U0001F60A", "Sad": "\U0001F622", "Stress": "\U0001F62B",
    "Angry": "\U0001F621", "Fear": "\U0001F628", "Neutral": "\U0001F610",
}

# GoEmotions ships 28 fine-grained labels. We map each one to the closest
# of our 6 app-level labels so the rest of the app (charts, DB columns,
# Streamlit UI) never has to know GoEmotions exists. Anything not listed
# here (e.g. "curiosity", "confusion") falls back to "Neutral".
GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy",
    "love": "Happy", "gratitude": "Happy", "optimism": "Happy",
    "relief": "Happy", "pride": "Happy", "admiration": "Happy",
    "approval": "Happy", "caring": "Happy",

    "sadness": "Sad", "disappointment": "Sad", "grief": "Sad",
    "remorse": "Sad",

    "nervousness": "Stress", "embarrassment": "Stress",
    "confusion": "Stress",

    "anger": "Angry", "annoyance": "Angry", "disgust": "Angry",
    "disapproval": "Angry",

    "fear": "Fear",

    "neutral": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "curiosity": "Neutral", "desire": "Neutral",
}


def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def _get_qwen():
    """Lazy-load Qwen2.5-0.5B-Instruct once per process (GPU if available).
    Still used by the wellness chatbot (wellness_chat_reply) -- only the
    emotion-detection step now uses BERT instead."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer


def _get_bert_emotion_pipeline():
    """
    Lazy-load the fine-tuned BERT emotion classifier once per process, using
    Hugging Face's `pipeline()` helper -- this bundles the tokenizer and the
    model together so we just call it with raw text and get scores back.

    `top_k=None` tells the pipeline to return a score for every label
    instead of just the single top prediction, so we can build a full
    scores dict (matching what the UI already expects).
    """
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification",
            model=BERT_EMOTION_MODEL_NAME,
            top_k=None,  # return all label scores, not just the best one
            device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline


def _bert_emotion(text: str) -> dict:
    """
    Classifies `text` using the fine-tuned BERT GoEmotions model, then maps
    the 28 GoEmotions labels down to our 6 app-level EMOTION_LABELS by
    summing mapped scores. Returns the same shape the rest of the app
    already expects: {"emotion": <label>, "scores": {label: 0-1, ...}}.
    """
    classifier = _get_bert_emotion_pipeline()

    if not text.strip():
        text = "(empty feedback)"

    # top_k=None returns a list of {"label": ..., "score": ...} dicts
    # covering all 28 GoEmotions labels, e.g.:
    # [{"label": "joy", "score": 0.82}, {"label": "neutral", "score": 0.05}, ...]
    raw_predictions = classifier(text, truncation=True)[0]

    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        goemotion_label = pred["label"].lower()
        app_label = GOEMOTIONS_TO_APP_LABEL.get(goemotion_label, "Neutral")
        app_scores[app_label] += pred["score"]

    # Normalize so scores are readable as relative proportions (0-1 range).
    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}

    final_emotion = max(app_scores, key=app_scores.get)
    return {"emotion": final_emotion, "scores": app_scores}


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive \U0001F60A"
    elif compound_score <= -0.05:
        final_sentiment = "Negative \U0001F614"
    else:
        final_sentiment = "Neutral \U0001F610"

    # --- Emotion detection via fine-tuned BERT (GoEmotions) ---
    # Replaces the earlier Qwen-prompting approach: BERT classifies
    # directly against fixed labels instead of generating free-form text
    # that then has to be parsed as JSON, so it's faster and more reliable.
    bert_result = _bert_emotion(translated_text)
    emotion_scores = bert_result["scores"]
    final_emotion_label = bert_result["emotion"]
    final_emotion = f"{final_emotion_label} {EMOTION_EMOJI.get(final_emotion_label, '')}"

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
    }


# --- Crisis-keyword safety net for the wellness chatbot -----------------
# This is a simple, deliberately blunt keyword check that runs regardless
# of what the LLM says. If it fires, we always show crisis resources —
# we never rely on the small LLM alone to catch something this important.
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)


def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)


def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    """
    Generates a supportive wellness chatbot reply using the Qwen chat model.
    (The chatbot still uses Qwen -- it needs to generate free-form
    conversational replies, which is a generation task, not a
    classification task, so BERT isn't a fit here.)

    `history` is an optional list of {"role": "user"|"assistant", "content": str}
    dicts representing prior turns in the conversation (kept short/recent by
    the caller — this function does not trim it).

    Always checks for crisis language first; if found, returns a fixed,
    resource-pointing message instead of an LLM-generated one, since we
    never want a small model improvising in a safety-critical moment.
    """
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}


Overwriting nlp_pipeline.py


In [32]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024  # 5 MB cap
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)  # exclude header from data row count

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1  # default to last column

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)  # just verifies the token; raises 401 if invalid

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results


class ChatTurn(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []


@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    """
    Wellness support chatbot endpoint. Stateless on the server: the client
    (Streamlit) sends the recent conversation history along with each new
    message, and we generate the next reply with the same Qwen model used
    for emotion detection.
    """
    get_user(authorization)  # verifies the token; raises 401 if invalid

    message = payload.message.strip()
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [turn.dict() for turn in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result


Overwriting backend.py


In [33]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

✅ Connected to PostgreSQL and ensured tables exist.


In [34]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)  # NLP libs (spaCy model etc.) take a little longer to import

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).")
print("Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://juicy-recognize-boundless.ngrok-free.dev" -> "http://localhost:8501"
FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).
Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.
